# Notebook 9: Mitra in place of TabPFN

The entry 24 places above us swapped TabPFN for **Mitra**, AutoGluon's tabular foundation
model, and reports: "Across seven matched embedding sources Mitra beat TabPFN on macro
held-out R-squared in six, and the two models' residuals correlated 0.94 to 0.98."

TabPFN is our strongest single arm and appears twice in `v21`, so an upgrade to it is an
upgrade to the component we lean on hardest. The 0.94 to 0.98 residual correlation means
to expect a *replacement*, not an addition: it should score better and add no diversity.

Same features as our best TabPFN arm (`emb_medium` + qHTS + primary screen, raw, no PCA),
same scaffold folds, so the output drops straight into the existing blend.

Control: a TabPFN arm on identical folds and features runs first and must land near
0.5881 / 0.7060 / 0.4448 / 0.8115, which is what notebook 1 measured. If it drifts, the
comparison is not measuring the model.

**AutoGluon is a heavy install and may force a runtime restart.** Everything writes to
Drive after each endpoint, so a restart costs one endpoint at most.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install autogluon.tabular[mitra] tabpfn==2.2.1

In [ ]:
import json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']; N_FOLDS=5
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy')
G=np.load(CACHE/'scaffold_groups.npy', allow_pickle=True)
EMB=np.load(CACHE/'emb_medium_inhib.npy').astype(np.float32)
EMB_TE=np.load(CACHE/'emb_medium_test.npy').astype(np.float32)
QH,QH_TE=np.load(CACHE/'qhts_tr.npy'),np.load(CACHE/'qhts_te.npy')
AUX,AUX_TE=np.load(CACHE/'aux_train.npy'),np.load(CACHE/'aux_test.npy')
X=np.nan_to_num(np.hstack([EMB,QH,AUX]).astype(np.float32))
X_TE=np.nan_to_num(np.hstack([EMB_TE,QH_TE,AUX_TE]).astype(np.float32))
COLS=[f'f{i}' for i in range(X.shape[1])]
REF={'CYP1A2':0.5881,'CYP2C9':0.7060,'CYP2D6':0.4448,'CYP3A4':0.8115}

def dev_index(iso):
    y=meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    o=np.isfinite(y)
    return y, np.flatnonzero(o & ~hold), np.flatnonzero(o & hold)

RESULTS=[]
def record(**kw):
    RESULTS.append(kw); json.dump(RESULTS, open(OUT/'results_nb9.json','w'), indent=2)
print('device', DEVICE, '| features', X.shape[1])

## Which tabular foundation models are actually available

In [ ]:
from autogluon.tabular.configs.hyperparameter_configs import get_hyperparameter_config
ag_hp = get_hyperparameter_config('zeroshot_2025_tabfm')
print('models in this config:', list(ag_hp))
MITRA_KEY = next((k for k in ag_hp if 'MITRA' in str(k).upper()), None)
print('Mitra key:', MITRA_KEY)
assert MITRA_KEY is not None, 'Mitra not present in this AutoGluon build; stop here'

In [ ]:
from autogluon.tabular import TabularPredictor
import shutil, warnings
warnings.filterwarnings('ignore')

def run(name, fit_predict):
    for iso in ISO:
        y, dev, ho = dev_index(iso)
        t0=time.time()
        oof=np.full(len(dev), np.nan)
        for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
            oof[va]=fit_predict(X[dev[tr]], y[dev[tr]], X[dev[va]])
        hp=fit_predict(X[dev], y[dev], X[ho])
        allr=np.concatenate([dev, ho])
        tp=fit_predict(X[allr], y[allr], X_TE)
        np.savez_compressed(OUT/f'{name}_{iso}.npz', oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        d=float(spearmanr(y[dev],oof).statistic); h=float(spearmanr(y[ho],hp).statistic)
        record(arm=name, iso=iso, dev_spearman=d, holdout_spearman=h,
               reference=REF[iso], seconds=round(time.time()-t0,1))
        print(f'  {name:8s} {iso}: dev {d:.4f} (notebook 1 got {REF[iso]:.4f}) | '
              f'holdout {h:.4f}  ({time.time()-t0:.0f}s)', flush=True)

def mitra(a, b, c):
    tr=pd.DataFrame(a, columns=COLS); tr['y']=b
    p=TabularPredictor(label='y', problem_type='regression', verbosity=0).fit(
        tr, hyperparameters={MITRA_KEY: ag_hp[MITRA_KEY]}, presets=None)
    out=p.predict(pd.DataFrame(c, columns=COLS)).to_numpy()
    shutil.rmtree(p.path, ignore_errors=True)
    return out

run('mitra', mitra)

## Control: TabPFN on identical folds and features

In [ ]:
from tabpfn import TabPFNRegressor
KW=dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
run('tabpfnctl', lambda a,b,c: TabPFNRegressor(**KW).fit(a,b).predict(c))

r=pd.DataFrame(RESULTS)
for col in ['dev_spearman','holdout_spearman']:
    t=r.pivot(index='arm', columns='iso', values=col); t['MACRO']=t.mean(axis=1)
    print(col); print(t.round(4).to_string()); print()
import shutil
from google.colab import files
shutil.make_archive('/content/notebook9_results','zip',OUT)
files.download('/content/notebook9_results.zip')